<div align="center">

# LOSO Model

</div>

In [10]:
from pathlib import Path
import h5py
import numpy as np

# 1. Load Files

In [16]:
# Seizure files
feature_dir = Path("Features/Seizure")

files = sorted(feature_dir.glob("chb01_*_seizure.mat"))

print(len(files))
for f in files:
    print(f.name)

7
chb01_03_seizure.mat
chb01_04_seizure.mat
chb01_15_seizure.mat
chb01_16_seizure.mat
chb01_18_seizure.mat
chb01_21_seizure.mat
chb01_26_seizure.mat


In [15]:
# Seizure files
feature_dir = Path("Features/Non_Seizure")

files = sorted(feature_dir.glob("chb01_*_features.mat"))

print(len(files))
for f in files:
    print(f.name)

30
chb01_01_features.mat
chb01_02_features.mat
chb01_05_features.mat
chb01_06_features.mat
chb01_07_features.mat
chb01_08_features.mat
chb01_09_features.mat
chb01_10_features.mat
chb01_11_features.mat
chb01_12_features.mat
chb01_13_features.mat
chb01_14_features.mat
chb01_17_features.mat
chb01_19_features.mat
chb01_20_features.mat
chb01_22_features.mat
chb01_23_features.mat
chb01_24_features.mat
chb01_25_features.mat
chb01_27_features.mat
chb01_29_features.mat
chb01_30_features.mat
chb01_31_features.mat
chb01_32_features.mat
chb01_33_features.mat
chb01_34_features.mat
chb01_36_features.mat
chb01_37_features.mat
chb01_38_features.mat
chb01_39_features.mat


In [10]:
# Inspeact one file
with h5py.File(files[0], "r") as f:
    print(list(f.keys()))
    print(f["all_feats"].shape)
    print(f["all_labels"].shape)

['all_feats', 'all_labels', 'bands', 'feat_len', 'fres', 'morder', 'step_samps', 'win_samps']
(1437, 1530)
(1, 1437)


In [ ]:
"""
Patient-specific seizure detection — rigorous nested Leave-One-Record-Out evaluation.

Design choices (and why):
  - Outer loop:  LeaveOneGroupOut by record   -> unbiased performance estimate.
  - Inner loop:  StratifiedGroupKFold by record -> hyperparameter tuning that keeps
                 records intact AND keeps seizures in every validation fold.
  - SelectKBest + StandardScaler live INSIDE a Pipeline, so they are refit on the
                 training side of every split -> no feature-selection / scaling leakage.
  - Model selection metric: AUPRC (average_precision) -> robust under ~1% prevalence.
  - Operating threshold is chosen on TRAINING out-of-fold scores to hit a target
                 false-detections-per-hour, then frozen and applied to the test record.
  - Held-out scores/events are POOLED across folds before metrics are computed,
                 because many records are single-class (seizure-free) and per-fold
                 segment metrics are undefined there.

Replace the two clearly marked integration points:
  (A) load_patient(...)              -> your feature matrices + record metadata
  (B) score_events_for_record(...)   -> a thin wrapper over YOUR event_level_scoring.py
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable

import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import (
    GridSearchCV,
    LeaveOneGroupOut,
    StratifiedGroupKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


# --------------------------------------------------------------------------- #
# Data containers
# --------------------------------------------------------------------------- #
@dataclass
class PatientData:
    X: np.ndarray                 # (n_windows, n_features)
    y: np.ndarray                 # (n_windows,)  in {0, 1}, 1 = ictal window
    record_id: np.ndarray         # (n_windows,)  group label per window
    window_step_s: float          # 2.5 for your 10 s / 75 % overlap setup
    # per-record annotations: record_id -> list of (onset_s, offset_s) seizures
    seizures: dict = field(default_factory=dict)
    # per-record interictal duration in HOURS used for the FD/h denominator
    interictal_hours: dict = field(default_factory=dict)


# --------------------------------------------------------------------------- #
# (A) INTEGRATION POINT — load your features. Stub shown for shape reference.
# --------------------------------------------------------------------------- #
def load_patient(patient_id: str) -> PatientData:
    raise NotImplementedError(
        "Return a PatientData built from your GC feature extractor. "
        "X/y/record_id must be aligned row-wise; record_id is the group key."
    )


# --------------------------------------------------------------------------- #
# (B) INTEGRATION POINT — wrap YOUR event_level_scoring.py
# --------------------------------------------------------------------------- #
def score_events_for_record(
    scores: np.ndarray,            # decision_function values for one record's windows
    threshold: float,
    seizures: list,                # [(onset_s, offset_s), ...] for this record
    window_step_s: float,
    k_consecutive: int,
    refractory_s: float,
) -> dict:
    """
    Expected to call into your event_level_scoring.py and return:
        {"n_true": int, "n_detected": int, "n_false": int,
         "latencies_s": list[float]}   # latencies only for true detections
    Implement as a thin adapter; the persistence + refractory + matching logic
    is the code you already have.
    """
    raise NotImplementedError("Adapt to your event_level_scoring.py interface.")


# --------------------------------------------------------------------------- #
# Pipeline + search space
# --------------------------------------------------------------------------- #
def build_pipeline() -> Pipeline:
    # Scaler first (SVM needs it); f_classif is scale-invariant so selection is
    # unaffected by ordering. Everything fits on the training split only.
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("select", SelectKBest(score_func=f_classif)),
            ("svc", SVC(kernel="rbf", class_weight="balanced", probability=False)),
        ]
    )


def param_grid(n_features: int) -> dict:
    k_choices = [k for k in (50, 100, 200, 400) if k <= n_features] + ["all"]
    return {
        "select__k": k_choices,
        "svc__C": [0.1, 1, 10, 100],
        "svc__gamma": ["scale", 1e-3, 1e-2, 1e-1],
        "svc__kernel": ["rbf", "linear"],  # linear baseline; drop if too slow
    }


def make_inner_cv(y_train: np.ndarray, groups_train: np.ndarray) -> StratifiedGroupKFold:
    """k=3..5 inner folds, capped by the number of seizure-bearing records so that
    every inner validation fold can contain at least one positive."""
    seizure_records = np.unique(groups_train[y_train == 1])
    k = int(np.clip(len(seizure_records), 2, 5))
    return StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=0)


# --------------------------------------------------------------------------- #
# Threshold selection on TRAINING out-of-fold scores (no test peeking)
# --------------------------------------------------------------------------- #
def select_threshold_for_fdr(
    oof_scores: np.ndarray,
    oof_record_id: np.ndarray,
    data: PatientData,
    target_fd_per_hour: float,
    k_consecutive: int,
    refractory_s: float,
    n_candidates: int = 200,
) -> float:
    """Sweep thresholds; for each, run the event scorer over the training records'
    out-of-fold scores; return the LOWEST (most sensitive) threshold whose pooled
    training FD/h is <= target. Lower threshold => higher sensitivity, so we want
    the most permissive operating point that still respects the false-alarm budget."""
    candidates = np.quantile(oof_scores, np.linspace(0.50, 0.999, n_candidates))
    train_records = np.unique(oof_record_id)
    total_train_hours = sum(data.interictal_hours.get(r, 0.0) for r in train_records)

    chosen = candidates[-1]  # default to the strictest if none meet the budget
    for thr in candidates:  # ascending => from permissive to strict
        n_false = 0
        for r in train_records:
            m = oof_record_id == r
            res = score_events_for_record(
                scores=oof_scores[m],
                threshold=thr,
                seizures=data.seizures.get(r, []),
                window_step_s=data.window_step_s,
                k_consecutive=k_consecutive,
                refractory_s=refractory_s,
            )
            n_false += res["n_false"]
        fd_per_hour = n_false / max(total_train_hours, 1e-9)
        if fd_per_hour <= target_fd_per_hour:
            chosen = thr
            break
    return float(chosen)


# --------------------------------------------------------------------------- #
# Nested LORO driver
# --------------------------------------------------------------------------- #
def nested_loro(
    data: PatientData,
    target_fd_per_hour: float = 1.0,
    k_consecutive: int = 4,        # Shoeb-aligned persistence
    refractory_s: float = 60.0,
    scoring: str = "average_precision",
) -> dict:
    outer = LeaveOneGroupOut()
    pooled_scores, pooled_y, pooled_records = [], [], []
    event_rows = []  # per held-out record event results

    for tr_idx, te_idx in outer.split(data.X, data.y, groups=data.record_id):
        X_tr, y_tr, g_tr = data.X[tr_idx], data.y[tr_idx], data.record_id[tr_idx]
        X_te = data.X[te_idx]
        test_record = data.record_id[te_idx][0]

        # --- inner hyperparameter search (training pool only) ---
        inner_cv = make_inner_cv(y_tr, g_tr)
        search = GridSearchCV(
            estimator=build_pipeline(),
            param_grid=param_grid(X_tr.shape[1]),
            scoring=scoring,
            cv=inner_cv.split(X_tr, y_tr, groups=g_tr),
            n_jobs=-1,
            refit=True,                       # refit best on full outer-train pool
        )
        search.fit(X_tr, y_tr)
        best = search.best_estimator_

        # --- out-of-fold training scores for threshold selection ---
        # Re-run the chosen pipeline through the inner CV to get honest OOF scores.
        oof_scores = np.full(len(y_tr), np.nan)
        for in_tr, in_va in inner_cv.split(X_tr, y_tr, groups=g_tr):
            fold_pipe = build_pipeline().set_params(**search.best_params_)
            fold_pipe.fit(X_tr[in_tr], y_tr[in_tr])
            oof_scores[in_va] = fold_pipe.decision_function(X_tr[in_va])
        valid = ~np.isnan(oof_scores)
        threshold = select_threshold_for_fdr(
            oof_scores[valid], g_tr[valid], data,
            target_fd_per_hour, k_consecutive, refractory_s,
        )

        # --- predict the held-out record and freeze the threshold ---
        test_scores = best.decision_function(X_te)
        pooled_scores.append(test_scores)
        pooled_y.append(data.y[te_idx])
        pooled_records.append(data.record_id[te_idx])

        ev = score_events_for_record(
            scores=test_scores,
            threshold=threshold,
            seizures=data.seizures.get(test_record, []),
            window_step_s=data.window_step_s,
            k_consecutive=k_consecutive,
            refractory_s=refractory_s,
        )
        ev["record"] = test_record
        ev["interictal_hours"] = data.interictal_hours.get(test_record, 0.0)
        event_rows.append(ev)

    return _pool_and_report(
        np.concatenate(pooled_scores),
        np.concatenate(pooled_y),
        event_rows,
    )


# --------------------------------------------------------------------------- #
# Pooled reporting (segment metrics on the pool, event metrics summed)
# --------------------------------------------------------------------------- #
def _pool_and_report(scores: np.ndarray, y: np.ndarray, event_rows: list) -> dict:
    seg = {}
    if len(np.unique(y)) == 2:        # both classes present in the pool
        seg["roc_auc"] = float(roc_auc_score(y, scores))
        seg["auprc"] = float(average_precision_score(y, scores))

    n_true = sum(r["n_true"] for r in event_rows)
    n_detected = sum(r["n_detected"] for r in event_rows)
    n_false = sum(r["n_false"] for r in event_rows)
    total_hours = sum(r["interictal_hours"] for r in event_rows)
    latencies = [l for r in event_rows for l in r["latencies_s"]]

    event = {
        "sensitivity": (n_detected / n_true) if n_true else float("nan"),
        "fd_per_hour": (n_false / total_hours) if total_hours else float("nan"),
        "median_latency_s": float(np.median(latencies)) if latencies else float("nan"),
        "n_seizures": n_true,
        "n_detected": n_detected,
        "n_false": n_false,
    }
    return {"segment": seg, "event": event, "per_record": event_rows}


# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    for patient_id in ["chb01"]:          # extend to more patients later
        data = load_patient(patient_id)   # (A)
        result = nested_loro(data, target_fd_per_hour=1.0, k_consecutive=4)
        print(patient_id, result["segment"], result["event"])